# Doppler Signal Inference

Generate synthetic Doppler radar data, train an MLP to predict target position from
multi-transmitter Doppler shift vectors, then visualize predicted vs actual Doppler maps.

In [ ]:
import sys
from pathlib import Path

# Ensure the package is importable when running from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bdl.datasets import DatasetAdapter, DopplerDataset
from bdl.loss import custom_doppler_loss
from bdl.inference import calculate_accuracy, create_analysis_image

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Generate Doppler Dataset

Each sample consists of:
- **Input**: 4 Doppler shift vectors (one per transmitter), each 1000 elements
- **Target**: 28x28 image with a single pixel marking the target position

The Doppler shift encodes target velocity and position relative to each transmitter.

In [ ]:
TRAIN_SAMPLES = 10000
VAL_SAMPLES = 512
BATCH_SIZE = 256


def collate_prebatched(batch):
    """Pass through pre-batched tensors from DatasetAdapter.__getitems__."""
    if isinstance(batch[0], torch.Tensor) and len(batch) >= 2:
        return batch[0], batch[1]
    elif isinstance(batch[0], (tuple, list)) and len(batch[0]) == 2:
        data, targets = zip(*batch)
        return torch.stack(data, dim=0), torch.stack(targets, dim=0)
    return batch


problem = DopplerDataset(device)
problem.create_dataset(TRAIN_SAMPLES, "train", use_cache=False)
problem.preprocess_data("train", normalize=True, add_noise=False, make_contiguous=True)
problem.create_dataset(VAL_SAMPLES, "validate", use_cache=False)
problem.preprocess_data("validate", normalize=True, add_noise=False, make_contiguous=True)

train_loader = DataLoader(
    DatasetAdapter(problem, TRAIN_SAMPLES, "train"),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    collate_fn=collate_prebatched,
)
val_loader = DataLoader(
    DatasetAdapter(problem, VAL_SAMPLES, "validate"),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    collate_fn=collate_prebatched,
)

print(f"Input shape: {problem.input_shape} -> {problem.input_size} features")
print(f"Output shape: {problem.output_shape} -> {problem.output_size} features")
print(f"Train: {TRAIN_SAMPLES}, Val: {VAL_SAMPLES}")

## Train Model

Simple MLP baseline trained with the custom Doppler loss function, which combines MSE,
peak intensity, spatial shift, and background suppression terms.

In [ ]:
INPUT_SIZE = problem.input_size   # 4000
OUTPUT_SIZE = problem.output_size  # 784
HIDDEN = 512
EPOCHS = 30
LR = 1e-3

model = nn.Sequential(
    nn.Linear(INPUT_SIZE, HIDDEN),
    nn.BatchNorm1d(HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, HIDDEN),
    nn.BatchNorm1d(HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, HIDDEN // 2),
    nn.BatchNorm1d(HIDDEN // 2),
    nn.ReLU(),
    nn.Linear(HIDDEN // 2, OUTPUT_SIZE),
    nn.Sigmoid(),
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_losses = []
val_losses = []
scaler = torch.GradScaler(device=device.type)

for epoch in (pbar := tqdm(range(EPOCHS))):
    # Train
    model.train()
    epoch_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type):
            outputs = model(inputs.view(inputs.size(0), -1))
            loss = custom_doppler_loss(outputs, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item() * inputs.size(0)
    scheduler.step()
    train_losses.append(epoch_loss / TRAIN_SAMPLES)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad(), torch.autocast(device_type=device.type):
        for inputs, targets in val_loader:
            outputs = model(inputs.view(inputs.size(0), -1))
            loss = custom_doppler_loss(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_losses.append(val_loss / VAL_SAMPLES)

    pbar.set_postfix(train=f"{train_losses[-1]:.4f}", val=f"{val_losses[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Doppler Loss")
ax.set_title("Training Loss")
ax.legend()
plt.tight_layout()
plt.show()

## Run Inference

Collect predictions on the validation set and reshape to 28x28 Doppler maps.

In [ ]:
model.eval()
all_outputs = []
all_targets = []

with torch.no_grad(), torch.autocast(device_type=device.type):
    for inputs, targets in tqdm(val_loader, desc="Inference"):
        outputs = model(inputs.view(inputs.size(0), -1))
        all_outputs.append(outputs.cpu())
        all_targets.append(targets.cpu())

outputs_np = torch.cat(all_outputs).numpy().reshape(VAL_SAMPLES, 28, 28).transpose(0, 2, 1)
targets_np = torch.cat(all_targets).numpy().reshape(VAL_SAMPLES, 28, 28).transpose(0, 2, 1)
indices = list(range(VAL_SAMPLES))

accuracy = calculate_accuracy(outputs_np, targets_np)
print(f"Accuracy: {accuracy:.2f}%")

## Visualize Results

Interactive browser for comparing predicted vs actual Doppler maps sample-by-sample.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

fig_browse = None
axs_browse = None
output_widget = widgets.Output()


def plot_results_page(page: int = 0, page_size: int = 1):
    global fig_browse, axs_browse

    start = page * page_size
    end = min(start + page_size, len(indices))
    current = indices[start:end]
    n = len(current)

    if fig_browse is None:
        fig_browse, axs_browse = plt.subplots(n, 3, figsize=(10, 3 * n))
    else:
        fig_browse.clear()
        fig_browse, axs_browse = plt.subplots(n, 3, figsize=(10, 3 * n), num=fig_browse.number)

    if n == 1:
        axs_browse = np.array([axs_browse])

    for i, idx in enumerate(current):
        pred = outputs_np[idx]
        actual = targets_np[idx]
        diff = actual - pred

        axs_browse[i, 0].imshow(pred, cmap="gray")
        axs_browse[i, 0].set_title(f"Predicted #{idx + 1}")
        fig_browse.colorbar(axs_browse[i, 0].images[0], ax=axs_browse[i, 0])

        axs_browse[i, 1].imshow(actual, cmap="gray")
        axs_browse[i, 1].set_title(f"Actual #{idx + 1}")
        fig_browse.colorbar(axs_browse[i, 1].images[0], ax=axs_browse[i, 1])

        axs_browse[i, 2].imshow(diff, cmap="bwr")
        axs_browse[i, 2].set_title(f"Difference #{idx + 1}")
        fig_browse.colorbar(axs_browse[i, 2].images[0], ax=axs_browse[i, 2])

    plt.tight_layout()
    fig_browse.canvas.draw_idle()


total_pages = len(indices)
page_slider = widgets.IntSlider(
    value=1, min=1, max=total_pages, step=1,
    description="Sample:", continuous_update=False,
)


def on_change(change):
    with output_widget:
        clear_output(wait=True)
        plot_results_page(page=change["new"] - 1)
        plt.show()
        print(f"Sample {change['new']} of {total_pages}")


page_slider.observe(on_change, names="value")
display(widgets.VBox([page_slider, output_widget]))
on_change({"new": 1})

## MSE Analysis

Aggregate error heatmap showing where the model struggles, and sample density across the target space.

In [ ]:
create_analysis_image(outputs_np, targets_np, indices, output_file="doppler_analysis.png")

from IPython.display import Image
Image("doppler_analysis.png")